In [1]:
!pip install seqeval -q

In [2]:
# Phase 1: Imports and Setup
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    pipeline,
)
from seqeval.metrics import (
    f1_score, accuracy_score, precision_score,
    recall_score, classification_report,
)
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

MODEL_CHECKPOINT = "bert-base-uncased"
BATCH_SIZE = 32           # larger batch for stability
MAX_LENGTH = 128
LEARNING_RATE = 3e-5
NUM_EPOCHS = 5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
FREEZE_LAYERS = 8         # freeze first 8/12 encoder layers

print(f"Model: {MODEL_CHECKPOINT} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}")

c:\Users\Majid Hussein\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Using device: cuda
Model: bert-base-uncased | Batch: 32 | LR: 3e-05


In [3]:
# Phase 2: Load Dataset and GROUP Labels
# The original dataset has 80+ fine-grained BIO tags which BERT struggles with.
# We group them into 7 core PII categories relevant to ZeroSec.

print("Loading dataset...")
dataset = load_dataset("ai4privacy/pii-masking-300k", split="train")

dataset = dataset.map(lambda x: {
    "tokens": x["mbert_text_tokens"],
    "labels": x["mbert_bio_labels"],
})

# Clean bad rows
def filter_bad_data(example):
    try:
        if not example["tokens"] or not example["labels"]:
            return False
        return len(example["tokens"]) == len(example["labels"])
    except:
        return False

dataset = dataset.filter(filter_bad_data)
print(f"Cleaned size: {len(dataset)}")

# --- GROUP fine-grained labels into core PII categories ---
LABEL_MAP = {
    # PERSON: names
    "GIVENNAME": "PERSON", "LASTNAME": "PERSON", "FIRSTNAME": "PERSON",
    "PREFIX": "PERSON", "TITLE": "PERSON", "USERNAME": "PERSON",
    # CONTACT: email, phone, social
    "EMAIL": "CONTACT", "PHONENUMBER": "CONTACT", "TEL": "CONTACT",
    "SOCIALNUM": "CONTACT", "SOCIALNUMBER": "CONTACT",
    # LOCATION: addresses, cities, countries
    "CITY": "LOCATION", "COUNTRY": "LOCATION", "STATE": "LOCATION",
    "STREET": "LOCATION", "STREETADDRESS": "LOCATION", "ZIPCODE": "LOCATION",
    "COUNTY": "LOCATION", "BUILDING": "LOCATION", "BUILDINGNUMBER": "LOCATION",
    "SECONDARYADDRESS": "LOCATION", "NEARBYGPSCOORDINATE": "LOCATION",
    # FINANCIAL: credit cards, bank, money
    "CREDITCARDNUMBER": "FINANCIAL", "CREDITCARDCVV": "FINANCIAL",
    "CARDISSUER": "FINANCIAL", "IBAN": "FINANCIAL", "BIC": "FINANCIAL",
    "BITCOINADDRESS": "FINANCIAL", "ETHEREUMADDRESS": "FINANCIAL",
    "CURRENCY": "FINANCIAL", "CURRENCYNAME": "FINANCIAL",
    "CURRENCYSYMBOL": "FINANCIAL", "CURRENCYCODE": "FINANCIAL",
    "ACCOUNTNUMBER": "FINANCIAL", "ACCOUNTNAME": "FINANCIAL",
    "AMOUNT": "FINANCIAL", "PIN": "FINANCIAL",
    # ID: government IDs, passports, SSNs
    "SSN": "ID", "PASSPORTNUMBER": "ID", "DRIVERLICENCE": "ID",
    "TAXIDENTIFICATIONNUMBER": "ID", "VEHICLEIDENTIFICATIONNUMBER": "ID",
    "VEHICLEVRM": "ID", "MASKEDNUMBER": "ID",
    # TEMPORAL: dates, times, ages
    "DATE": "TEMPORAL", "TIME": "TEMPORAL", "DOB": "TEMPORAL",
    "DATEOFBIRTH": "TEMPORAL", "AGE": "TEMPORAL",
    # DIGITAL: IPs, URLs, passwords, MACs
    "IP": "DIGITAL", "IPV4": "DIGITAL", "IPV6": "DIGITAL",
    "URL": "DIGITAL", "MAC": "DIGITAL", "MACADDRESS": "DIGITAL",
    "PASSWORD": "DIGITAL", "USERAGENT": "DIGITAL",
    "IMEI": "DIGITAL", "LITECOINADDRESS": "DIGITAL",
}

def group_label(label):
    """Map fine-grained BIO label to grouped BIO label."""
    if label == "O":
        return "O"
    prefix = label[:2]  # B- or I-
    entity = label[2:]  # e.g. GIVENNAME
    grouped = LABEL_MAP.get(entity, None)
    if grouped is None:
        return "O"  # unknown entity types become O
    return prefix + grouped

def group_labels_fn(example):
    example["labels"] = [group_label(l) for l in example["labels"]]
    return example

dataset = dataset.map(group_labels_fn)

# Build label vocab from grouped labels
unique_labels = sorted(set(l for seq in dataset["labels"] for l in seq))
label2id = {l: i for i, l in enumerate(unique_labels)}
id2label = {i: l for l, i in label2id.items()}
print(f"\nGrouped labels ({len(unique_labels)}): {unique_labels}")

# Encode to integers
dataset = dataset.map(lambda x: {"labels": [label2id[l] for l in x["labels"]]}, batched=False)

# Split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(dataset['train'])} | Test: {len(dataset['test'])}")

Loading dataset...
Cleaned size: 177652

Grouped labels (13): ['B-CONTACT', 'B-DIGITAL', 'B-FINANCIAL', 'B-LOCATION', 'B-PERSON', 'B-TEMPORAL', 'I-CONTACT', 'I-DIGITAL', 'I-FINANCIAL', 'I-LOCATION', 'I-PERSON', 'I-TEMPORAL', 'O']
Train: 159886 | Test: 17766


In [4]:
# Phase 3: Tokenization with Label Alignment
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
    )
    all_labels = []
    for i, labels in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev_word_idx:
                label_ids.append(labels[word_idx])
            else:
                # For sub-word tokens: use I- tag if original is B-, else same
                label_ids.append(labels[word_idx])
            prev_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

print("Tokenizing...")
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True, batch_size=1000)
print("Done.")

Tokenizing...


Map: 100%|██████████| 17766/17766 [00:34<00:00, 522.30 examples/s]

Done.


In [5]:
# Phase 4: Model Setup with Layer Freezing + Class Weights
from collections import Counter

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_preds = [
        [id2label[pr] for pr, la in zip(pred, lab) if la != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[la] for pr, la in zip(pred, lab) if la != -100]
        for pred, lab in zip(predictions, labels)
    ]
    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
        "accuracy": accuracy_score(true_labels, true_preds),
    }

# --- Compute class weights to handle imbalance ---
print("Computing class weights...")
label_counts = Counter()
for seq in dataset["train"]["labels"]:
    label_counts.update(seq)

total = sum(label_counts.values())
n_classes = len(unique_labels)
class_weights = []
for i in range(n_classes):
    count = label_counts.get(i, 1)
    # Inverse frequency weight, capped to avoid extreme values
    weight = min(total / (n_classes * count), 10.0)
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class weights (O={class_weights[label2id['O']]:.2f}):")
for label, idx in sorted(label2id.items()):
    if label != "O":
        print(f"  {label}: {class_weights[idx]:.2f}")

# --- Custom Trainer with weighted loss ---
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
        loss = loss_fn(logits.view(-1, n_classes), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# --- Load model ---
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id,
)

# Freeze first N encoder layers
modules_to_freeze = [model.bert.embeddings, *model.bert.encoder.layer[:FREEZE_LAYERS]]
for module in modules_to_freeze:
    for param in module.parameters():
        param.requires_grad = False

total_p = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params: {total_p:,} | Trainable: {trainable_p:,} ({trainable_p/total_p:.1%})")
print(f"Frozen: {FREEZE_LAYERS}/12 encoder layers + embeddings")

model.to(device)
print("Model ready.")

Computing class weights...
Class weights (O=0.09):
  B-CONTACT: 10.00
  B-DIGITAL: 10.00
  B-FINANCIAL: 10.00
  B-LOCATION: 10.00
  B-PERSON: 10.00
  B-TEMPORAL: 10.00
  I-CONTACT: 1.60
  I-DIGITAL: 2.43
  I-FINANCIAL: 10.00
  I-LOCATION: 3.30
  I-PERSON: 4.75
  I-TEMPORAL: 6.36


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Total params: 108,901,645 | Trainable: 28,361,485 (26.0%)
Frozen: 8/12 encoder layers + embeddings
Model ready.


In [ ]:
# Phase 5: Training with Weighted Loss
data_collator = DataCollatorForTokenClassification(tokenizer)

args = TrainingArguments(
    output_dir="./pii_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to="none",
)

# Use WeightedTrainer instead of Trainer
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training with weighted loss...")
trainer.train()

Starting training with weighted loss...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.737000,0.678378,0.043209,0.279104,0.074833,0.692741
2,0.631900,0.578774,0.063001,0.352730,0.106908,0.778203


KeyboardInterrupt: 

: 

In [ ]:
# Phase 6: Evaluation, Save, and Demo
print("Final Evaluation...")
metrics = trainer.evaluate()
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

# Save model
model_path = "./bert_pii_model_final"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print(f"\nModel saved to {model_path}")

# Demo
print("\n--- Inference Demo ---")
pipe = pipeline("token-classification", model=model_path, aggregation_strategy="simple", device=0 if torch.cuda.is_available() else -1)

test_texts = [
    "My name is John Doe and my email is john.doe@example.com",
    "Credit card number 4111-1111-1111-1111 expires 12/2025",
    "SSN: 123-45-6789, born on 1990-05-15 in New York",
    "Contact me at +1-555-123-4567 or visit https://example.com",
]

for text in test_texts:
    print(f"\nInput: {text}")
    results = pipe(text)
    for r in results:
        print(f"  Found: '{r['word']}' -> {r['entity_group']} ({r['score']:.4f})")